# Join Optimisation

## Overview
Joins are the most common source of performance problems in analytical queries. The database planner chooses a join algorithm and join order based on table statistics; understanding how this works helps you write queries the planner can optimise, and helps you recognise when to intervene.

**The three join algorithms:**

| Algorithm | How it works | Best when |
|---|---|---|
| **Nested Loop** | For each row in outer, scan inner | Inner is small or indexed; outer is small |
| **Hash Join** | Build hash table on smaller input; probe with larger | Neither input is sorted; one fits in memory |
| **Merge Join** | Both inputs sorted on join key; advance pointers | Both inputs already sorted or can be cheaply sorted |

**Join order matters:**
The planner tries to minimise the number of rows flowing between plan nodes. Joining the most selective (smallest result) tables first reduces intermediate row counts. With N tables, there are N! possible join orders; the planner uses cost estimates to pick the best.

**Key join optimisation rules:**
1. Ensure join columns are indexed (especially FK columns)
2. Filter early -- push WHERE conditions into CTEs/subqueries before joining
3. Avoid joining on expressions: `JOIN ON UPPER(a.name) = UPPER(b.name)` prevents index use
4. Pre-aggregate before joining to prevent row fan-out

**PostgreSQL notes:** `SET join_collapse_limit = 1` forces the planner to use the join order as written (useful for testing). `SET enable_hashjoin = OFF` disables Hash Join to force a different algorithm. Always reset these after testing.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE sites (site_id INTEGER PRIMARY KEY, site_name TEXT NOT NULL, region TEXT, latitude REAL, longitude REAL, elevation_m REAL, established_date TEXT, active INTEGER DEFAULT 1);CREATE TABLE species (species_id INTEGER PRIMARY KEY, common_name TEXT NOT NULL, scientific_name TEXT NOT NULL UNIQUE, taxon_group TEXT, native INTEGER DEFAULT 1, at_risk INTEGER DEFAULT 0);CREATE TABLE field_crew (crew_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, role TEXT, certified INTEGER DEFAULT 1);CREATE TABLE observations (obs_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), species_id INTEGER REFERENCES species(species_id), crew_id INTEGER REFERENCES field_crew(crew_id), obs_date TEXT NOT NULL, count_individuals INTEGER, life_stage TEXT, method TEXT, notes TEXT);CREATE TABLE water_quality (wq_id INTEGER PRIMARY KEY, site_id INTEGER REFERENCES sites(site_id), crew_id INTEGER REFERENCES field_crew(crew_id), sample_date TEXT NOT NULL, ph REAL, dissolved_o2 REAL, turbidity_ntu REAL, temp_c REAL, conductivity_us REAL);INSERT INTO sites VALUES (1,'Fundy Estuary','Atlantic',45.78,-64.52,3.2,'2018-04-01',1),(2,'Kejimkujik Lake','Atlantic',44.43,-65.21,156.0,'2017-06-15',1),(3,'Presqu ile Point','Great Lakes',43.99,-77.72,75.5,'2019-03-20',1),(4,'Rondeau Bay','Great Lakes',42.31,-81.87,176.0,'2018-09-10',1),(5,'Athabasca Delta','Boreal',58.72,-110.87,213.0,'2016-01-15',1),(6,'Wapusk Tundra','Boreal',57.92,-93.15,45.0,'2017-07-01',1),(7,'Clayoquot Sound','Pacific',49.15,-125.93,12.0,'2015-05-20',1),(8,'Boundary Bay','Pacific',49.01,-122.97,1.5,'2016-08-01',1),(9,'Lac Saint-Pierre','St Lawrence',46.19,-72.87,8.0,'2018-02-14',1),(10,'Baie des Chaleurs','Atlantic',48.01,-65.72,5.0,'2019-11-01',0);INSERT INTO species VALUES (1,'Atlantic Salmon','Salmo salar','Fish',1,1),(2,'Great Blue Heron','Ardea herodias','Bird',1,0),(3,'Wood Duck','Aix sponsa','Bird',1,0),(4,'Spotted Turtle','Clemmys guttata','Reptile',1,1),(5,'Common Loon','Gavia immer','Bird',1,0),(6,'Muskrat','Ondatra zibethicus','Mammal',1,0),(7,'Northern Pike','Esox lucius','Fish',1,0),(8,'Bald Eagle','Haliaeetus leucocephalus','Bird',1,0),(9,'American Bittern','Botaurus lentiginosus','Bird',1,1),(10,'Snapping Turtle','Chelydra serpentina','Reptile',1,0),(11,'Rainbow Trout','Oncorhynchus mykiss','Fish',0,0),(12,'Ring-necked Duck','Aythya collaris','Bird',1,0),(13,'Beaver','Castor canadensis','Mammal',1,0),(14,'River Otter','Lontra canadensis','Mammal',1,0),(15,'Painted Turtle','Chrysemys picta','Reptile',1,0);INSERT INTO field_crew VALUES (1,'Dr. Aroha Ngata','Biologist',1),(2,'Liam Chen','Technician',1),(3,'Fatima Al-Rashid','Biologist',1),(4,'James MacLeod','Technician',1),(5,'Sofia Petrov','Student',0);INSERT INTO observations VALUES (1,1,1,1,'2023-04-10',12,'Adult','Electrofishing',NULL),(2,1,2,2,'2023-04-10',3,'Adult','Visual',NULL),(3,2,5,1,'2023-04-15',2,'Adult','Auditory',NULL),(4,2,4,3,'2023-04-15',8,'Juvenile','Visual',NULL),(5,3,2,2,'2023-05-01',5,'Adult','Visual',NULL),(6,3,3,4,'2023-05-01',7,'Adult','Visual',NULL),(7,4,10,3,'2023-05-10',2,'Adult','Visual',NULL),(8,4,7,1,'2023-05-10',4,'Adult','Electrofishing',NULL),(9,5,13,2,'2023-05-20',6,'Adult','Camera Trap',NULL),(10,5,14,3,'2023-05-20',1,'Adult','Visual',NULL),(11,6,8,1,'2023-06-01',3,'Adult','Visual',NULL),(12,6,6,4,'2023-06-01',9,'Adult','Trap',NULL),(13,7,14,3,'2023-06-15',2,'Adult','Visual',NULL),(14,7,5,2,'2023-06-15',4,'Adult','Auditory',NULL),(15,8,2,1,'2023-07-01',12,'Adult','Visual',NULL),(16,8,8,4,'2023-07-01',2,'Adult','Visual',NULL),(17,9,1,3,'2023-07-10',7,'Adult','Electrofishing',NULL),(18,9,9,1,'2023-07-10',1,'Adult','Visual','First sighting this season'),(19,1,4,2,'2023-08-05',1,'Adult','Visual',NULL),(20,2,13,4,'2023-08-05',4,'Adult','Camera Trap',NULL),(21,3,6,3,'2023-08-12',11,'Adult','Trap',NULL),(22,4,2,2,'2023-08-12',7,'Adult','Visual',NULL),(23,5,8,1,'2023-09-01',1,'Adult','Visual',NULL),(24,6,14,3,'2023-09-01',3,'Adult','Visual',NULL),(25,7,1,4,'2023-09-15',18,'Adult','Electrofishing',NULL),(26,8,5,2,'2023-09-15',6,'Adult','Visual',NULL),(27,9,4,3,'2023-09-20',2,'Juvenile','Visual',NULL),(28,1,7,1,'2023-10-01',3,'Adult','Electrofishing',NULL),(29,2,1,2,'2023-10-01',9,'Adult','Electrofishing',NULL),(30,3,8,4,'2023-10-10',4,'Adult','Visual',NULL),(31,4,5,1,'2023-10-15',3,'Adult','Auditory',NULL),(32,5,2,3,'2023-10-15',8,'Adult','Visual',NULL);INSERT INTO water_quality VALUES (1,1,1,'2023-04-10',7.2,9.1,3.4,8.5,142.0),(2,1,2,'2023-06-15',7.4,8.6,4.2,14.2,138.0),(3,1,3,'2023-08-20',7.1,7.8,5.1,19.6,145.0),(4,2,1,'2023-04-15',6.8,10.2,1.2,7.1,98.0),(5,2,4,'2023-07-01',6.9,9.4,1.8,16.3,102.0),(6,3,2,'2023-05-01',7.6,9.8,2.3,11.4,188.0),(7,3,3,'2023-08-01',7.5,8.2,3.7,20.1,192.0),(8,4,1,'2023-05-10',7.8,9.5,1.9,13.0,205.0),(9,4,4,'2023-09-05',7.7,8.9,2.5,18.4,198.0),(10,5,2,'2023-05-20',7.3,10.8,0.8,9.2,55.0),(11,5,1,'2023-09-10',7.2,9.6,1.1,15.7,58.0),(12,6,3,'2023-06-01',6.5,11.2,0.5,6.8,32.0),(13,7,2,'2023-06-15',7.9,9.0,1.4,12.5,310.0),(14,8,4,'2023-07-01',7.8,8.7,2.1,17.2,295.0),(15,9,1,'2023-07-10',7.6,9.3,2.8,14.8,178.0);""")
conn.commit()
print("Ecological schema ready.")

---
## Join plans: SQLite EXPLAIN QUERY PLAN

In [ ]:
# Add indexes to show their effect on join plans
conn.executescript("""
    CREATE INDEX idx_obs_site    ON observations (site_id);
    CREATE INDEX idx_obs_species ON observations (species_id);
    CREATE INDEX idx_obs_crew    ON observations (crew_id);
    CREATE INDEX idx_wq_site     ON water_quality (site_id);
""")
conn.commit()

print("=== Two-table join: observations + sites ===")
plan = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT o.obs_id, s.site_name, o.obs_date
    FROM   observations AS o
    JOIN   sites        AS s ON o.site_id = s.site_id
    WHERE  s.region = 'Atlantic'
""").fetchall()
for r in plan:
    print(" ", r)

print()
print("=== Three-table join: observations + sites + species ===")
plan2 = conn.execute("""
    EXPLAIN QUERY PLAN
    SELECT o.obs_id, s.site_name, sp.common_name, o.count_individuals
    FROM   observations AS o
    JOIN   sites        AS s  ON o.site_id    = s.site_id
    JOIN   species      AS sp ON o.species_id = sp.species_id
    WHERE  s.region = 'Boreal'
      AND  sp.taxon_group = 'Bird'
""").fetchall()
for r in plan2:
    print(" ", r)


---
## Fan-out: joining before aggregating

In [ ]:
print("=== Fan-out problem: aggregate AFTER join ===")
q_bad = """
SELECT  s.site_name,
        COUNT(o.obs_id)        AS observation_count,
        SUM(o.count_individuals) AS total_individuals
FROM    sites        AS s
JOIN    observations AS o ON s.site_id = o.site_id
JOIN    water_quality AS wq ON s.site_id = wq.site_id
GROUP BY s.site_name
ORDER BY s.site_name
"""
df_bad = pd.read_sql(q_bad, conn)
print("Result joining obs and wq before aggregating:")
print(df_bad.to_string(index=False))
print()
print("Problem: each observation joins to EVERY wq record for the same site.")
print("Site 1 has 5 obs and 3 wq records -> 15 rows before GROUP BY.")
print("COUNT and SUM are inflated.")

print()
print("=== Fix: pre-aggregate each table before joining ===")
q_good = """
WITH obs_summary AS (
    SELECT site_id,
           COUNT(*)             AS observation_count,
           SUM(count_individuals) AS total_individuals
    FROM   observations GROUP BY site_id
),
wq_summary AS (
    SELECT site_id, ROUND(AVG(ph),2) AS avg_ph, COUNT(*) AS wq_samples
    FROM   water_quality GROUP BY site_id
)
SELECT  s.site_name,
        COALESCE(o.observation_count, 0)  AS observations,
        COALESCE(o.total_individuals, 0)  AS individuals,
        wq.avg_ph,
        COALESCE(wq.wq_samples, 0)        AS wq_samples
FROM    sites       AS s
LEFT JOIN obs_summary AS o  ON s.site_id = o.site_id
LEFT JOIN wq_summary  AS wq ON s.site_id = wq.site_id
ORDER BY s.site_name
"""
df_good = pd.read_sql(q_good, conn)
print("Correct result with pre-aggregation:")
print(df_good.to_string(index=False))


---
## PostgreSQL join algorithm hints and tuning

In [ ]:
print("=== PostgreSQL join algorithm comparison ===")
algorithms = [
    ("Nested Loop",
     "Best for: small outer, indexed inner.
Worst for: large outer, unindexed inner (O(n*m) reads).
Hint: SET enable_hashjoin=OFF; SET enable_mergejoin=OFF;"),
    ("Hash Join",
     "Best for: large unsorted inputs; when one side fits in work_mem.
Worst for: when hash table spills to disk (Batches > 1).
Hint: SET enable_nestloop=OFF; SET enable_mergejoin=OFF;
Tune: SET work_mem = '256MB'; -- increase hash table memory"),
    ("Merge Join",
     "Best for: both inputs pre-sorted on join key (e.g. both ordered by date).
Worst for: when sort cost exceeds hash cost.
Hint: SET enable_nestloop=OFF; SET enable_hashjoin=OFF;"),
]
for name, notes in algorithms:
    print(f"--- {name} ---")
    print(notes)
    print()

print("=== Join order forcing in PostgreSQL ===")
print("""
  -- Force join order as written (disable reordering):
  SET join_collapse_limit = 1;

  -- Rewrite with explicit FROM order to test performance:
  -- Smaller/more selective table first
  SELECT ...
  FROM   species AS sp            -- small: 15 rows
  JOIN   observations AS o        -- large: 32 rows
         ON o.species_id = sp.species_id
  JOIN   sites AS s               -- medium: 10 rows
         ON o.site_id = s.site_id
  WHERE  sp.at_risk = 1;

  -- Always reset after testing:
  RESET join_collapse_limit;
""")
conn.close()


---
## Common Pitfalls

**1. Joining before aggregating (fan-out)**
Joining observations (32 rows per site) to water_quality (3 rows per site) produces 96 rows per site before GROUP BY. COUNT and SUM run on the inflated row count, giving wrong answers. Always pre-aggregate each table in a CTE before joining when combining two many-to-one relationships against the same parent.

**2. Joining on expressions instead of raw columns**
`JOIN ON UPPER(a.species_name) = UPPER(b.species_name)` cannot use an index on `species_name` -- the index stores raw values, not UPPER values. Either normalise data at insert time (store names in uppercase consistently) or create a functional index: `CREATE INDEX ON species (UPPER(common_name))`.

**3. Implicit Cartesian products from missing JOIN conditions**
`FROM observations, sites WHERE observations.obs_date > '2023-06-01'` without a join condition produces all combinations of observations x sites (320 rows from 32 x 10). Always use explicit `JOIN ... ON ...` syntax; it makes missing conditions a syntax error rather than a silent result explosion.

**4. Unindexed FK columns forcing full scans on the child table**
`JOIN observations ON observations.site_id = sites.site_id` with no index on `observations.site_id` requires a full scan of observations for every row in sites. PostgreSQL does not auto-create indexes on FK columns. Add them explicitly.

**5. Over-relying on JOIN order hints**
`SET join_collapse_limit = 1` fixes the join order but bypasses the planner's cost model. After schema changes or data growth, the forced order may no longer be optimal. Use hints only for short-term diagnosis; the real fix is fresh statistics (`ANALYZE`) and proper indexes.

**6. Forgetting to reset enable_* planner settings**
`SET enable_hashjoin = OFF` in a psql session or connection pool disables Hash Join for all subsequent queries in that connection. This can cause severe performance regressions for other queries. Always follow testing overrides with `RESET enable_hashjoin` or `SET enable_hashjoin = ON`.


---
*sql_methods_library - Samantha McGarrigle*